In [ ]:
# Prep v2 at 9 slice groups instead of 5, for test-time averaging over slice
# windows. CPU-only, internet off: does not touch the GPU quota.
import glob, os, shutil, sys, time

GIT_SHA = 'e443b25'

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

from knee.dicom import SLICE_GROUP, SLOTS
from knee.prep import load_study_npz, prep_slots, save_study_npz
import inspect
# A src without window_groups preps without complaint and fails only at the last
# cell an hour later, as the first run of these shards did for a different reason.
assert 'window_groups' in inspect.signature(prep_slots).parameters, 'rsna-knee-src predates window_groups'
print('knee package imported from', PKG, '| GIT_SHA', GIT_SHA)
print('slots:', [name for name, _, _ in SLOTS])

In [ ]:
# Sharding by contiguous index range over sorted study UIDs, identical to the
# phase6 shards, so shard i here holds exactly the studies of phase6 shard i and
# the last cell can compare them one to one.
PILOT_N = None
SHARD_INDEX = 0       # per-shard kernels differ only in this line
N_SHARDS = 4

# The one change from prep v2. Nine groups instead of five; groups 0,2,4,6,8 are
# the five the checkpoints trained on, the other four sit halfway between them.
# Sized before launching: 1.80x prep v2 per study on the local 20-study sample,
# so ~17 GB over the corpus and ~4.3 GB per shard, inside the 20 GB output cap.
N_GROUPS = 9
# The intensity window comes from the five training groups and is applied to all
# nine. The first run took it from all 27 selected slices, which rescaled the
# shared groups by a mean 1.2 grey levels (max 67) and failed the comparison with
# prep v2 in the last cell.
WINDOW_GROUPS = 5
CROP_MM = 130.0
OUT_SIZE = 336
SLOT_SCHEME = 'v2'

OUT_DIR = '/kaggle/working/prepped'
os.makedirs(OUT_DIR, exist_ok=True)

import math
import numpy as np
import pandas as pd

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
series_df = pd.read_csv(f'{COMP_DIR}/train_series.csv')

gold_cols = [c for c in train_df.columns if c not in ('StudyInstanceUID', 'Report')]
gold_uids = set(train_df.loc[train_df[gold_cols].notna().any(axis=1), 'StudyInstanceUID'])

TRAIN_SERIES_DIR = f'{COMP_DIR}/train_series'
all_study_uids = sorted(
    d for d in os.listdir(TRAIN_SERIES_DIR)
    if os.path.isdir(os.path.join(TRAIN_SERIES_DIR, d))
)

per_shard = math.ceil(len(all_study_uids) / N_SHARDS)
lo = SHARD_INDEX * per_shard
study_uids = all_study_uids[lo:lo + per_shard]

if PILOT_N is not None:
    stride = max(len(study_uids) // PILOT_N, 1)
    sampled = study_uids[::stride][:PILOT_N]
    study_uids = sorted(set(sampled) | {u for u in study_uids if u in gold_uids})

print(f'{len(all_study_uids)} studies on disk, {len(gold_uids)} gold-labeled')
print(f'shard {SHARD_INDEX}/{N_SHARDS}: [{lo}, {lo + per_shard}) -> {len(study_uids)} studies '
      f'({sum(u in gold_uids for u in study_uids)} gold)')

In [ ]:
meta_rows = []
failed_studies = []
t0 = time.time()

for i, study_uid in enumerate(study_uids):
    try:
        slot_slices, meta = prep_slots(
            study_uid,
            TRAIN_SERIES_DIR,
            series_df,
            n_groups=N_GROUPS,
            window_groups=WINDOW_GROUPS,
            crop_mm=CROP_MM,
            out_size=OUT_SIZE,
            slot_scheme=SLOT_SCHEME,
            is_gold=study_uid in gold_uids,
        )
        save_study_npz(os.path.join(OUT_DIR, f'{study_uid}.npz'), slot_slices, meta)
        meta_rows.append(meta)
    except Exception as exc:
        # Broad on purpose, as in every prep shard: one malformed study must not
        # abort a multi-hour shard and lose every artifact with it.
        failed_studies.append({'StudyInstanceUID': study_uid,
                               'error': f'{type(exc).__name__}: {exc}'})
    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        print(f'{i + 1}/{len(study_uids)} prepped, {elapsed:.0f}s '
              f'({elapsed / (i + 1) * 1000:.0f} ms/study)')

elapsed = time.time() - t0
print(f'done: {len(meta_rows)} prepped, {len(failed_studies)} failed, {elapsed:.0f}s')
for row in failed_studies[:10]:
    print(row)
assert meta_rows, 'no study prepped -- every gate below would report on an empty set'

In [ ]:
series_meta_rows = []
for m in meta_rows:
    for series_uid, sm in m['series'].items():
        series_meta_rows.append({'StudyInstanceUID': m['StudyInstanceUID'],
                                 'SeriesInstanceUID': series_uid, **sm})
series_meta_df = pd.DataFrame(series_meta_rows)

mm_per_px = series_meta_df['mm_per_px'].unique()
assert len(mm_per_px) == 1, f'physical scale is not constant: {mm_per_px}'
print(f'PASS: {len(series_meta_df)} stored series, all at {mm_per_px[0]:.4f} mm/px')

sizes_mb = np.array([os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6
                     for f in os.listdir(OUT_DIR) if f.endswith('.npz')])
print(f'MB/study mean {sizes_mb.mean():.2f}, p95 {np.percentile(sizes_mb, 95):.2f}, '
      f'max {sizes_mb.max():.2f} -> shard total {sizes_mb.sum() / 1000:.2f} GB')

In [ ]:
manifest = pd.DataFrame([
    {
        'StudyInstanceUID': m['StudyInstanceUID'],
        'side': m['side'],
        'route': m['route'],
        'is_gold': m['is_gold'],
        'PatientSex': m['PatientSex'],
        'n_series_on_disk': m['n_series_on_disk'],
        'n_slots_filled': len(m['series']),
        'slots_filled': '|'.join(sorted(sm['slot'] for sm in m['series'].values())),
        'n_slices_stored': sum(sm['n_slices_stored'] for sm in m['series'].values()),
        'n_decode_failures': len(m['decode_failures']),
    }
    for m in meta_rows
])
manifest.to_csv(f'/kaggle/working/prep_manifest_shard{SHARD_INDEX}.csv', index=False)
series_meta_df.to_csv(f'/kaggle/working/prep_series_meta_shard{SHARD_INDEX}.csv', index=False)
pd.DataFrame(failed_studies, columns=['StudyInstanceUID', 'error']).to_csv(
    f'/kaggle/working/prep_failed_shard{SHARD_INDEX}.csv', index=False)
print(f'{len(os.listdir(OUT_DIR))} artifacts in {OUT_DIR}; manifests written')

In [ ]:
# GATE -- these artifacts must contain prep v2's. Slot selection does not depend
# on N_GROUPS, so every study must file the same slots as its phase6 artifact;
# and groups 0,2,4,6,8 here must be the five groups the confirm checkpoints
# trained on. Last cell on purpose: the manifests are already written, so a
# failure costs the check, not the shard.
#
# Compared as decoded pixels. Identical crops through the same JPEG encoder give
# identical bytes, but phase6 ran on an older Kaggle image, so an encoder update
# could move pixels by a level or two. That would not change what the model sees
# in any way that matters, while a wrong group pick moves them by whole-slice
# amounts. Both numbers are printed; the assert is on the mean.
v2_manifests = glob.glob(f'/kaggle/input/**/prep_manifest_shard{SHARD_INDEX}.csv', recursive=True)
assert len(v2_manifests) == 1, f'expected the phase6 shard {SHARD_INDEX} manifest: {v2_manifests}'
v2_manifest = pd.read_csv(v2_manifests[0]).set_index('StudyInstanceUID')
V2_DIR = os.path.join(os.path.dirname(v2_manifests[0]), 'prepped')

here = manifest.set_index('StudyInstanceUID')
common = here.index.intersection(v2_manifest.index)
print(f'{len(common)} of {len(here)} studies also in phase6 shard {SHARD_INDEX} '
      f'({len(v2_manifest)} there)')
moved = int((here.loc[common, 'slots_filled'].fillna('')
             != v2_manifest.loc[common, 'slots_filled'].fillna('')).sum())
print(f'studies filing different slots than prep v2: {moved}')
assert moved == 0, 'slot selection changed with N_GROUPS'

slot_names = [name for name, _, _ in SLOTS]
worst, total_abs, n_px = 0, 0, 0
sample = list(common)[:200]
for uid in sample:
    nine, _ = load_study_npz(os.path.join(OUT_DIR, f'{uid}.npz'), order=slot_names)
    five, _ = load_study_npz(os.path.join(V2_DIR, f'{uid}.npz'), order=slot_names)
    for name, slices in five.items():
        a5 = np.stack(slices).astype(np.int16)
        a9 = np.stack(nine[name]).astype(np.int16).reshape(N_GROUPS, SLICE_GROUP, *a5.shape[1:])
        diff = np.abs(a9[0::2].reshape(a5.shape) - a5)
        worst = max(worst, int(diff.max()))
        total_abs += int(diff.sum())
        n_px += diff.size
mean_abs = total_abs / max(n_px, 1)
print(f'groups 0,2,4,6,8 vs prep v2 over {len(sample)} studies: '
      f'mean |diff| {mean_abs:.4f}, max |diff| {worst}')
assert mean_abs < 0.5, 'the 5-of-9 read does not reproduce prep v2\'s groups'